# Music Annotation Project — Notebook 38
## V8.2i Envelope Relaxation and Perceptual Severity Calibration — Amendment 38.4

**Runtime:** Colab CPU. Run all cells in order.

**Status:** development-only and single-reviewer exploratory. No retraining, recalibration, threshold adoption, challenger freeze, independent-test nomination or production authority.

### Amendment 38.4
Notebook 37's persisted threshold-page directories are resolved using their actual fixed filenames:

- `threshold_candidate.png` — corrected page for that candidate threshold
- `v8_2_registered.png` — registered 0.995 comparator
- `v7_8.png` — frozen V7.8 page
- `observed.png` — observed source page
- `difference_from_registered_v8_2.png` — candidate-versus-registered difference
- `newly_preserved_vs_0995_overlay.png` — newly-preserved overlay

Adjacent-threshold A/B pairs use `threshold_candidate.png` from the corresponding threshold directories. When the higher threshold is 0.995, `v8_2_registered.png` from the lower-threshold directory is used. No page reconstruction is performed.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q "gradio>=4.44,<6" pandas numpy pyarrow pillow matplotlib


## 1. Configuration


In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import base64,hashlib,itertools,json,os,shutil,zipfile
import numpy as np,pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
NB37_ROOT=ROOT/'observed_print_classifier_v8_2h_threshold_frontier'
OUT_ROOT=ROOT/'observed_print_classifier_v8_2i_envelope_relaxation'
SOURCE_NB37_RUN_OVERRIDE=None
THRESHOLDS=[.995,.990,.985,.980,.975,.950,.925,.900,.875,.850,.825,.800,.750,.700,.600,.500]
SEED=38001; REVIEWER='reviewer_01'
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_2i_38_3';OUT=OUT_ROOT/RUN;TABLES=OUT/'tables';AUDIT=OUT/'audit';REVIEW=OUT/'review';RENDER=OUT/'rendered_threshold_pages';UPLOAD=OUT/'upload'
for p in [OUT,TABLES,AUDIT,REVIEW,RENDER]:p.mkdir(parents=True,exist_ok=False)
print('Output:',OUT)


## 2. Helpers


In [ ]:
def now():return datetime.now(timezone.utc).isoformat()
def sha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def csv(d,p):p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);d.to_csv(p,index=False)
def js(p,x):Path(p).write_text(json.dumps(x,indent=2,sort_keys=True,default=str))
def stop(stage,msg,details=None):js(AUDIT/f'failure_{stage}.json',{'stage':stage,'message':msg,'details':details or {},'utc':now()});raise RuntimeError(f'{stage}: {msg}')
def col(d,names,required=True):
 z={str(x).lower():x for x in d.columns}
 for n in names:
  if n.lower() in z:return z[n.lower()]
 if required:raise KeyError(f'Missing one of {names}; columns={list(d.columns)}')
def image_ok(p):
 try:
  with Image.open(p) as im:im.verify()
  return True
 except:return False
def read_gray(p):return np.array(Image.open(p).convert('L'))
def write_gray(p,a):Path(p).parent.mkdir(parents=True,exist_ok=True);Image.fromarray(np.asarray(a,dtype=np.uint8),'L').save(p)
def candidates(root):return sorted([p for p in Path(root).iterdir() if p.is_dir()],key=lambda p:p.stat().st_mtime) if Path(root).exists() else []


## 3. Resolve Notebook 37 and lock directly consumed files


In [ ]:
REQ=['experiment_identity.json','source_v82_record.json','tables/threshold_frontier_summary.csv','tables/threshold_frontier_by_page.csv','tables/threshold_frontier_by_source_group.csv','tables/incremental_threshold_pixels.parquet','tables/incremental_component_review_queue_all.csv','tables/catastrophic_component_first_appearance_inventory.csv','upload/upload_manifest.csv']
hits=[p for p in candidates(NB37_ROOT) if all((p/r).exists() for r in REQ)]
NB37=Path(SOURCE_NB37_RUN_OVERRIDE) if SOURCE_NB37_RUN_OVERRIDE else (hits[-1] if hits else None)
if NB37 is None:stop('input','No complete Notebook 37 run found')
locks=[]
for r in REQ:
 p=NB37/r
 if not p.is_file():stop('input','Required file missing',{'file':r})
 locks.append({'relative_path':r,'absolute_path':str(p.resolve()),'bytes':p.stat().st_size,'sha256':sha(p)})
source_lock=pd.DataFrame(locks);csv(source_lock,AUDIT/'notebook38_source_input_lock.csv');print('Notebook 37:',NB37)


## 4. Package-manifest audit — warning only for package-copy discrepancies


In [ ]:
m=pd.read_csv(NB37/'upload/upload_manifest.csv');pc=col(m,['relative_path','path','file','filename']);hc=col(m,['sha256','sha_256','hash'],False);rows=[]
for _,r in m.iterrows():
 raw=str(r[pc]);q=Path(raw);opts=[NB37/'upload'/q,NB37/q]
 if 'upload' in q.parts:
  k=list(q.parts).index('upload');opts.insert(0,NB37/'upload'/Path(*list(q.parts)[k+1:]))
 p=next((x for x in opts if x.is_file()),None);actual=sha(p) if p else '';expected=str(r[hc]).strip() if hc and pd.notna(r[hc]) else '';match=(actual.lower()==expected.lower()) if p and expected else np.nan;status='verified_package_copy' if p and (not expected or match is True) else ('package_copy_hash_mismatch' if p else 'unresolved_package_row');rows.append({'manifest_path':raw,'resolved':str(p or ''),'expected_sha256':expected,'actual_sha256':actual,'status':status})
ma=pd.DataFrame(rows);csv(ma,AUDIT/'nb37_manifest_verification.csv');summary={'verified':int(ma.status.eq('verified_package_copy').sum()),'warnings':int(ma.status.ne('verified_package_copy').sum()),'scope':'curated package only'};js(AUDIT/'nb37_manifest_verification_summary.json',summary);print(summary)


## 5. Registered threshold and gate replay


In [ ]:
front=pd.read_csv(NB37/'tables/threshold_frontier_summary.csv');actual=[round(float(x),3) for x in front[col(front,['threshold'])]]
if actual!=THRESHOLDS:stop('thresholds','Threshold order mismatch',{'actual':actual})
fields=['threshold','true_print','false_annotation','selected_print_pixels','utility','catastrophic_components','false_true_ratio','false_preservation_rate','benefiting_pages','benefiting_source_groups','D3_catastrophic_safety','D4_false_preservation','D5_pixel_safety','D6_positive_utility','D7_nontrivial_benefit','D8_source_stability']
missing=[x for x in fields if x not in front]
if missing:stop('gates','Missing frontier fields',{'missing':missing})
csv(front[fields],TABLES/'registered_D3_D8_results.csv');print(front[fields].to_string(index=False))


## 6. Build stable transition-review units


In [ ]:
q=pd.read_csv(NB37/'tables/incremental_component_review_queue_all.csv');cf=pd.read_csv(NB37/'tables/catastrophic_component_first_appearance_inventory.csv');u=pd.DataFrame()
for target,names in {'page':['page'],'source_group':['source_group'],'component':['mask_component_id','component_id'],'hi':['higher_threshold','from_threshold','high_threshold'],'lo':['lower_threshold','to_threshold','low_threshold'],'new_tp':['new_true_print','incremental_true_print','true_print'],'new_fp':['new_false_annotation','incremental_false_annotation','false_annotation'],'local':['local_panel','local_crop','local_path'],'wide':['wide_panel','wide_crop','wide_path'],'locator':['locator','whole_page_locator','locator_path'],'truth_overlay':['truth_overlay','cumulative_truth_overlay','truth_overlay_path']}.items():
 x=col(q,names,target in ['page','source_group','component','hi','lo'])
 u[target]=q[x] if x is not None else ''
u.page=u.page.astype(str);u.source_group=u.source_group.astype(str);u.component=pd.to_numeric(u.component).astype(int);u.hi=pd.to_numeric(u.hi).round(3);u.lo=pd.to_numeric(u.lo).round(3);u['review_unit_id']=u.apply(lambda r:'RU_'+hashlib.sha256(f'{SEED}|{r.page}|{r.component}|{r.hi}|{r.lo}'.encode()).hexdigest()[:16],axis=1)
g=u.groupby('source_group',as_index=False).agg(units=('review_unit_id','size'),pages=('page','nunique')).sort_values('units',ascending=False);target=len(u)/2;cal=[];n=0
for x in g.source_group:
 if n<target:cal.append(x);n+=int(g.loc[g.source_group.eq(x),'units'].iloc[0])
g['split']=np.where(g.source_group.isin(cal),'rubric_calibration','audit_reserve');u=u.merge(g[['source_group','split']],on='source_group');csv(g,TABLES/'review_split_map.csv');csv(u,TABLES/'review_unit_manifest.csv');print('Review units:',len(u))


## 7. Resolve persisted Notebook 37 threshold-page directories

This stage uses Notebook 37's fixed persisted filenames directly. It does not infer a page from generic image names and does not reconstruct a candidate from pixel coordinates.

For each review transition:

1. resolve the lower-threshold directory containing `threshold_candidate.png`;
2. resolve the higher-threshold directory and its `threshold_candidate.png`;
3. when the higher threshold is 0.995, use `v8_2_registered.png` from the lower-threshold directory;
4. attach the persisted difference, overlay, V7.8 and observed images as post-lock diagnostics;
5. verify readability, equal page shape and SHA-256 identity before review.


In [ ]:
PERSISTED_NAMES={
    'candidate':'threshold_candidate.png',
    'registered':'v8_2_registered.png',
    'v78':'v7_8.png',
    'observed':'observed.png',
    'difference':'difference_from_registered_v8_2.png',
    'overlay':'newly_preserved_vs_0995_overlay.png',
}

# Index every actual Notebook 37 candidate directory, including packaged copies.
candidate_dirs=sorted({p.parent.resolve() for p in NB37.rglob(PERSISTED_NAMES['candidate'])})
if not candidate_dirs:
    stop('review_assets','No persisted threshold_candidate.png files were found under Notebook 37',
         {'notebook37_run':str(NB37)})

def page_aliases(page):
    s=str(page); stem=Path(s).stem; aliases={s.lower(),stem.lower()}
    digits=''.join(ch for ch in stem if ch.isdigit())
    if digits:
        n=int(digits)
        aliases.update({str(n),f'{n:02d}',f'{n:03d}',f'page{n}',f'page{n:02d}',f'page{n:03d}',
                        f'page_{n}',f'page_{n:02d}',f'page_{n:03d}'})
    return aliases

def threshold_markers(t):
    s=f'{float(t):.3f}'
    return {s,s.replace('.','_'),s.replace('.',''),f't{s}',f't_{s}',f'threshold_{s}',
            f'threshold_{s.replace(".","_")}',f'p{s}',f'p_{s}'}

def path_score(directory,page,t,anchors=()):
    parts=[x.lower() for x in directory.parts]
    joined='/'.join(parts)
    pscore=max([4 if a in parts else 2 if a in joined else 0 for a in page_aliases(page)] or [0])
    tscore=max([5 if m in parts else 3 if m in joined else 0 for m in threshold_markers(t)] or [0])
    ascore=0
    for a in anchors:
        try:
            ap=Path(str(a))
            if ap.exists():
                ar=ap.resolve()
                if directory==ar.parent or directory in ar.parents: ascore=max(ascore,8)
            name=ap.name
            if name and any(x.name==name for x in directory.rglob(name)): ascore=max(ascore,4)
        except Exception:
            pass
    return pscore+tscore+ascore,pscore,tscore,ascore

def resolve_dir(page,t,row):
    anchors=[row.get(x,'') for x in ['local','wide','locator','truth_overlay'] if str(row.get(x,'')).strip()]
    ranked=[]
    for d in candidate_dirs:
        score,ps,ts,ans=path_score(d,page,t,anchors)
        if ps>0 and ts>0: ranked.append((score,ps,ts,ans,d))
    if not ranked:
        return None,[],f'no directory matched page={page}, threshold={t:.3f}'
    ranked=sorted(ranked,key=lambda x:(x[0],x[1],x[2],x[3],-len(x[4].parts)),reverse=True)
    top=ranked[0][0]
    top_dirs=sorted({x[4] for x in ranked if x[0]==top},key=lambda x:str(x))
    if len(top_dirs)!=1:
        return None,ranked[:10],f'ambiguous top score {top} across {len(top_dirs)} directories'
    return top_dirs[0],ranked[:10],''

def require_persisted(directory,name):
    p=Path(directory)/name
    return p if p.is_file() and image_ok(p) else None

rows=[];audit_rows=[];failures=[]
for _,r in u.iterrows():
    lo=float(r.lo);hi=float(r.hi)
    lo_dir,lo_rank,lo_error=resolve_dir(r.page,lo,r)
    hi_dir=None;hi_rank=[];hi_error=''
    if np.isclose(hi,.995):
        hi_dir=lo_dir
    else:
        hi_dir,hi_rank,hi_error=resolve_dir(r.page,hi,r)
    low=require_persisted(lo_dir,PERSISTED_NAMES['candidate']) if lo_dir else None
    if np.isclose(hi,.995):
        high=require_persisted(lo_dir,PERSISTED_NAMES['registered']) if lo_dir else None
        high_role='v8_2_registered.png at 0.995'
    else:
        high=require_persisted(hi_dir,PERSISTED_NAMES['candidate']) if hi_dir else None
        high_role='threshold_candidate.png'
    diagnostics={k:require_persisted(lo_dir,v) if lo_dir else None for k,v in PERSISTED_NAMES.items()
                 if k not in ['candidate']}
    reasons=[]
    if lo_error:reasons.append('lower: '+lo_error)
    if hi_error:reasons.append('higher: '+hi_error)
    if low is None:reasons.append('lower threshold_candidate.png missing/unreadable')
    if high is None:reasons.append('higher page missing/unreadable: '+high_role)
    if low and high:
        with Image.open(low) as a,Image.open(high) as b:
            if a.size!=b.size:reasons.append(f'page shape mismatch: lower={a.size}, higher={b.size}')
    audit_rows.append({'review_unit_id':r.review_unit_id,'page':r.page,'higher_threshold':hi,
        'lower_threshold':lo,'higher_role':high_role,'higher_directory':str(hi_dir or ''),
        'lower_directory':str(lo_dir or ''),'higher_path':str(high or ''),'lower_path':str(low or ''),
        'higher_sha256':sha(high) if high else '','lower_sha256':sha(low) if low else '',
        'status':'failed' if reasons else 'verified','reason':' | '.join(reasons),
        'higher_top_candidates':json.dumps([str(x[4]) for x in hi_rank[:5]]),
        'lower_top_candidates':json.dumps([str(x[4]) for x in lo_rank[:5]])})
    if reasons:
        failures.append(audit_rows[-1]);continue
    left_low=int(hashlib.sha256(f'{SEED}|{r.review_unit_id}'.encode()).hexdigest(),16)%2==0
    A,B=(low,high) if left_low else (high,low)
    mapping={'A':'lower','B':'higher'} if left_low else {'A':'higher','B':'lower'}
    rows.append({**r.to_dict(),'whole_page_A':str(A),'whole_page_B':str(B),
        'A_sha256':sha(A),'B_sha256':sha(B),'A_method':'persisted_notebook37_page',
        'B_method':'persisted_notebook37_page','randomisation_side_locked':base64.urlsafe_b64encode(json.dumps(mapping).encode()).decode(),
        # post-lock diagnostics use the persisted Notebook 37 files directly
        'wide':str(diagnostics.get('overlay') or ''),'local':str(diagnostics.get('difference') or ''),
        'locator':str(diagnostics.get('v78') or ''),'truth_overlay':str(diagnostics.get('observed') or ''),
        'registered_path':str(diagnostics.get('registered') or ''),
        'overlay_path':str(diagnostics.get('overlay') or ''),
        'difference_path':str(diagnostics.get('difference') or ''),
        'v78_path':str(diagnostics.get('v78') or ''),'observed_path':str(diagnostics.get('observed') or '')})

asset_audit=pd.DataFrame(audit_rows)
csv(asset_audit,AUDIT/'whole_page_asset_resolution.csv')
csv(pd.DataFrame(failures),AUDIT/'missing_review_assets.csv')
if failures:
    stop('review_assets','Persisted Notebook 37 page resolution failed',
         {'count':len(failures),'examples':failures[:10],
          'expected_filenames':PERSISTED_NAMES,'indexed_candidate_directories':len(candidate_dirs)})
blind=pd.DataFrame(rows)
csv(blind,TABLES/'blinded_review_manifest.csv')
print('Indexed persisted candidate directories:',len(candidate_dirs))
print('Verified blinded adjacent-threshold pairs:',len(blind))
print(asset_audit.status.value_counts())


## 8. Gradio review


In [ ]:
import gradio as gr
R1=REVIEW/'perceptual_review_round1.csv';R2=REVIEW/'diagnostic_review_round2.csv';C1=['review_unit_id','visibility','preference','harm','benefit','scope','musical_risk','prominence','action','confidence','note','utc','locked'];C2=['review_unit_id','mechanism','resolved_harm','resolved_benefit','note','utc']
def readr(p,cols):return pd.read_csv(p,keep_default_na=False) if Path(p).exists() else pd.DataFrame(columns=cols)
def saver(p,cols,d):z=readr(p,cols);z=z[z.review_unit_id.astype(str)!=str(d['review_unit_id'])];csv(pd.concat([z,pd.DataFrame([d])])[cols],p)
def pending():return [i for i,x in enumerate(blind.review_unit_id) if str(x) not in set(readr(R2,C2).review_unit_id.astype(str))]
def view(i):
 i=int(i);r=blind.iloc[i];return i,r.whole_page_A,r.whole_page_B,f'{i+1}/{len(blind)}',r.review_unit_id
def first(i,*v):
 if any(x in ['',None] for x in v[:-1]):raise gr.Error('Complete all fields')
 r=blind.iloc[int(i)];d=dict(zip(C1[1:-2],v[:-1]));d.update({'review_unit_id':r.review_unit_id,'note':v[-1],'utc':now(),'locked':True});saver(R1,C1,d);return gr.update(visible=False),gr.update(visible=True),r.get('wide',''),r.get('local',''),r.get('locator',''),r.get('truth_overlay','')
def second(i,mech,h,b,note):
 r=blind.iloc[int(i)];saver(R2,C2,{'review_unit_id':r.review_unit_id,'mechanism':mech,'resolved_harm':h,'resolved_benefit':b,'note':note,'utc':now()});p=pending();return view(p[0] if p else int(i))+ (gr.update(visible=True),gr.update(visible=False),)
with gr.Blocks() as app:
 state=gr.State(pending()[0] if pending() else 0);unit=gr.Textbox();status=gr.Markdown()
 with gr.Row():A=gr.Image(type='filepath',label='A');B=gr.Image(type='filepath',label='B')
 with gr.Column(visible=True) as p1:
  vis=gr.Radio(['no difference','visible difference','unclear']);pref=gr.Radio(['A','B','neither','mixed','unclear']);harm=gr.Radio(['P0','P1','P2','P3','P4','U']);benefit=gr.Radio(['B0','B1','B2','B3','B4','U']);scope=gr.Dropdown(['pixel-fragment','component-local','staff/system','page-repeated','page-global']);risk=gr.Dropdown(['none','staff','notehead','stem/beam','accidental','rest','articulation','dynamic','text/lyric','barline/repeat','other','unclear']);prom=gr.Radio(['none','faint','moderate','prominent']);action=gr.Radio(['accept','accept-with-note','manual-correct','reject-policy-case','unclear']);conf=gr.Radio(['low','medium','high']);note=gr.Textbox();b1=gr.Button('Lock first pass')
 with gr.Column(visible=False) as p2:
  with gr.Row():wide=gr.Image();local=gr.Image();locator=gr.Image();truth=gr.Image()
  mech=gr.Dropdown(['correct print recovery','retained annotation','mixed overlap','registration ambiguity','truth-boundary sensitivity','display artefact','other']);rh=gr.Radio(['P0','P1','P2','P3','P4','U']);rb=gr.Radio(['B0','B1','B2','B3','B4','U']);dn=gr.Textbox();b2=gr.Button('Save and next')
 app.load(view,[state],[state,A,B,status,unit]);b1.click(first,[state,vis,pref,harm,benefit,scope,risk,prom,action,conf,note],[p1,p2,wide,local,locator,truth]);b2.click(second,[state,mech,rh,rb,dn],[state,A,B,status,unit,p1,p2])
app.launch(share=True,debug=False)


## 9. Completion and exploratory summary


In [ ]:
r1=readr(R1,C1);r2=readr(R2,C2);expected=set(blind.review_unit_id.astype(str));done=set(r2.review_unit_id.astype(str));csv(pd.DataFrame({'review_unit_id':sorted(expected),'complete':[x in done for x in sorted(expected)]}),AUDIT/'review_coverage.csv')
if expected!=done:stop('review_completion','Review incomplete',{'expected':len(expected),'complete':len(done)})
joined=blind.merge(r1,on='review_unit_id').merge(r2,on='review_unit_id');hn={'P0':0,'P1':1,'P2':2,'P3':3,'P4':4};joined['harm_n']=joined.resolved_harm.map(hn)
summary=[]
for t in THRESHOLDS:
 z=joined[joined.lo.astype(float).ge(t)];summary.append({'threshold':t,'review_units':len(z),'P2_plus':int(z.harm_n.ge(2).sum()),'P3_plus':int(z.harm_n.ge(3).sum()),'P4':int(z.harm_n.eq(4).sum()),'hard_stop':bool(z.harm_n.eq(4).any())})
result=pd.DataFrame(summary);csv(result,TABLES/'perceptual_D3_D8_results.csv');csv(result,TABLES/'envelope_relaxation_frontier.csv');status='single-reviewer-exploratory-with-provenance-warning' if summary and ma.status.ne('verified_package_copy').any() else 'single-reviewer-exploratory';verdict={'status':status,'threshold_adoption_authority':False,'production_authority':False,'source_lock_sha256':hashlib.sha256(source_lock.to_csv(index=False).encode()).hexdigest()};js(OUT/'notebook38_verdict.json',verdict);Path(OUT/'executive_summary.txt').write_text(json.dumps(verdict,indent=2));print(result.to_string(index=False));print(verdict)


## 10. Final package


In [ ]:
items=[OUT/'notebook38_verdict.json',OUT/'executive_summary.txt',TABLES/'registered_D3_D8_results.csv',TABLES/'review_unit_manifest.csv',TABLES/'review_split_map.csv',TABLES/'blinded_review_manifest.csv',TABLES/'perceptual_D3_D8_results.csv',TABLES/'envelope_relaxation_frontier.csv',R1,R2,AUDIT/'notebook38_source_input_lock.csv',AUDIT/'nb37_manifest_verification.csv',AUDIT/'nb37_manifest_verification_summary.json',AUDIT/'whole_page_asset_resolution.csv',AUDIT/'missing_review_assets.csv',AUDIT/'review_coverage.csv']
missing=[str(p) for p in items if not Path(p).exists()]
if missing:stop('package','Missing outputs',{'missing':missing})
UPLOAD.mkdir();rows=[]
for src in items:
 rel=Path(src).relative_to(OUT);dst=UPLOAD/rel;dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(src,dst);rows.append({'relative_path':rel.as_posix(),'sha256':sha(dst),'bytes':dst.stat().st_size})
csv(pd.DataFrame(rows),UPLOAD/'upload_manifest.csv');bundle=OUT/'notebook38_upload_bundle.zip'
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
 for p in UPLOAD.rglob('*'):
  if p.is_file():z.write(p,arcname=(Path('upload')/p.relative_to(UPLOAD)).as_posix())
print(bundle,sha(bundle))


## Interpretation boundary

The review pages are Notebook 37's persisted artefacts. `threshold_candidate.png` supplies each candidate-threshold page; `v8_2_registered.png` supplies the 0.995 endpoint; `difference_from_registered_v8_2.png`, `newly_preserved_vs_0995_overlay.png`, `v7_8.png` and `observed.png` are post-lock diagnostics. No page reconstruction or model rescoring occurs. Results remain single-reviewer exploratory.
